In [8]:
import subprocess

from typing import Annotated, TypedDict, List

from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.graph import StateGraph, END


In [9]:
def get_github_token() -> str | None:
    """Возвращает персональный токен GitHub, используя GitHub CLI.

    Вместо того чтобы хранить токен в коде или переменных окружения,
    мы делегируем его получение инструменту ``gh`` (GitHub CLI).
    Это безопасно: токен никогда не попадает в файлы проекта.

    Returns:
        Строка с токеном если GitHub CLI установлен и пользователь
        аутентифицирован (``gh auth login``), иначе ``None``.

    Raises:
        Не бросает исключений — все ошибки перехватываются внутри.
    """
    try:
        token = subprocess.check_output(["gh", "auth", "token"]).decode("utf-8").strip()
        return token
    except Exception as e:
        print("Ошибка: Убедитесь, что GitHub CLI установлен и вы залогинены (gh auth login)")
        return None


In [ ]:
@tool
def multiply(a: int, b: int) -> int:
    """Умножает два целых числа.

    Это наш первый tool для демонстрации tool-calling в агенте.

    Args:
        a: Первый множитель.
        b: Второй множитель.

    Returns:
        Результат умножения `a * b`.

    Example:
        >>> multiply.invoke({"a": 17, "b": 23})
        391
    """
    return a * b


class AgentState(TypedDict):
    """Схема состояния, которое передается между узлами графа.

    Attributes:
        messages: История сообщений агента.
            `add_messages` сообщает LangGraph, что новые сообщения нужно
            добавлять в список, а не перезаписывать полностью.
    """

    messages: Annotated[List[BaseMessage], add_messages]


# Создаем LLM-клиент и «биндим» к нему доступные инструменты.
# bind_tools(...) позволяет модели возвращать tool_calls, которые затем
# перехватываются ToolNode и выполняются в графе.
llm = ChatOpenAI(
    base_url="https://models.inference.ai.azure.com",
    api_key=get_github_token(),
    model="Llama-3.3-70B-Instruct",
    temperature=0,
).bind_tools([multiply])


def llm_node(state: AgentState) -> dict:
    """Узел LLM: анализирует историю и решает, нужен ли вызов tool.

    Если модель решает использовать инструмент, она вернет AIMessage
    с `tool_calls`. Если инструмент не нужен, вернет обычный текстовый ответ.

    Args:
        state: Текущее состояние графа с историей сообщений.

    Returns:
        Частичное обновление состояния в формате LangGraph:
        словарь с ключом `messages` и новым сообщением модели.

    Example:
        >>> out = llm_node({"messages": [HumanMessage(content="17*23")]})
        >>> out.keys()
        dict_keys(["messages"])
    """
    response = llm.invoke(state["messages"])
    return {"messages": response}


# ToolNode исполняет вызовы инструментов, которые LLM запросила через tool_calls.
tools_node = ToolNode([multiply])


In [ ]:
# =============================================================================
# Сборка графа
# -----------------------------------------------------------------------------
# Архитектура:
#   [START] -> llm:
#       --(если tool_calls)--> tools -> llm -> [END]
#       --(если tool не нужен)---------------> [END]
#
# tools_condition автоматически проверяет, есть ли в последнем сообщении
# вызовы инструментов. Если есть — идем в узел tools, иначе завершаем.
# =============================================================================
graph = StateGraph(AgentState)
graph.add_node("llm", llm_node)
graph.add_node("tools", tools_node)
graph.set_entry_point("llm")
graph.add_conditional_edges("llm", tools_condition, {"tools": "tools", END: END})
graph.add_edge("tools", "llm")

app = graph.compile()
app


In [19]:
# =============================================================================
# Обычный запуск (invoke): получаем финальное состояние после завершения графа
# =============================================================================
result = app.invoke(
    {
        "messages": [
            SystemMessage(content="Ты tool-using ассистент. Всегда отвечай с использованием данных тебе tools."),
            HumanMessage(content="Сколько будет если умножить 17 на 23?"),
        ]
    }
)

# Последнее сообщение — итоговый ответ модели после возможных tool-вызовов.
print(result["messages"][-1].content)


Ответ: 391


In [20]:
# =============================================================================
# Потоковый запуск (stream): смотрим пошагово, как граф проходит узлы
# -----------------------------------------------------------------------------
# stream_mode="values" отдает срез состояния после каждого шага.
# Это удобно для дебага:
# - видно, когда модель запросила tool_calls
# - видно ответ инструмента
# - видно финальный ответ модели
# =============================================================================
for step in app.stream(
    {
        "messages": [
            SystemMessage(content="Ты tool-using ассистент. Всегда отвечай с использованием данных тебе tools."),
            HumanMessage(content="Сколько будет если умножить 17 на 23?"),
        ]
    },
    stream_mode="values",
):
    last = step["messages"][-1]

    print("NODE OUTPUT:")
    print(type(last).__name__)

    if hasattr(last, "content"):
        print("Content:", last.content)

    if hasattr(last, "tool_calls") and last.tool_calls:
        print("Tool calls:", last.tool_calls)

    print("==========")

NODE OUTPUT:
HumanMessage
Content: Сколько будет если умножить 17 на 23?
NODE OUTPUT:
AIMessage
Content: 
Tool calls: [{'name': 'multiply', 'args': {'a': '17', 'b': '23'}, 'id': 'call_2fb612092f6340f996f14afe', 'type': 'tool_call'}]
NODE OUTPUT:
ToolMessage
Content: 391
NODE OUTPUT:
AIMessage
Content: Результат умножения 17 на 23 равен 391.
